In [1]:
import pandas as pd
import numpy as np
import ast

IN_PATH = "Claude_multi_step.csv"
OUT_PATH = "Claude_multi_step_extracted.csv"

PRED_COL = "predicted"
GT_COL   = "ground_truth"

KEYS = ["TOTALDEMAND", "RRP"]


def safe_to_dict(x):
    """Convert a cell to dict safely (handles dict, stringified dict, NaN)."""
    if isinstance(x, dict):
        return x
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return {}
        try:
            val = ast.literal_eval(x)
            return val if isinstance(val, dict) else {}
        except Exception:
            return {}
    return {}


def extract_series(d, key):
    """
    Extract all numeric first elements from dict[key] where dict[key] can be:
      - [value, ts]
      - [[value, ts]]
      - [[v1, ts], [v2, ts], ...]
    Returns: list[float] (possibly empty)
    """
    if not isinstance(d, dict) or key not in d:
        return []

    v = d[key]

    # Case A: [value, ts]
    if isinstance(v, (list, tuple)) and len(v) >= 1 and not isinstance(v[0], (list, tuple)):
        # v[0] is the numeric value
        try:
            return [float(v[0])]
        except Exception:
            return []

    # Case B/C: [[value, ts], [value, ts], ...]
    if isinstance(v, (list, tuple)):
        out = []
        for item in v:
            if isinstance(item, (list, tuple)) and len(item) >= 1:
                try:
                    out.append(float(item[0]))
                except Exception:
                    # skip non-numeric
                    pass
        return out

    return []


def extract_timestamp_series(d, key):
    """
    Optional: extract all timestamps (2nd element) if present.
    Returns: list[str]
    """
    if not isinstance(d, dict) or key not in d:
        return []

    v = d[key]

    # [value, ts]
    if isinstance(v, (list, tuple)) and len(v) >= 2 and not isinstance(v[0], (list, tuple)):
        return [str(v[1])]

    # [[value, ts], ...]
    if isinstance(v, (list, tuple)):
        out = []
        for item in v:
            if isinstance(item, (list, tuple)) and len(item) >= 2:
                out.append(str(item[1]))
        return out

    return []


# ----------------------------
# Load
# ----------------------------
df = pd.read_csv(IN_PATH)

# Ensure dicts
df[PRED_COL] = df[PRED_COL].apply(safe_to_dict)
df[GT_COL]   = df[GT_COL].apply(safe_to_dict)

# ----------------------------
# Extract series (values)
# ----------------------------
for k in KEYS:
    df[f"pred_{k}_series"] = df[PRED_COL].apply(lambda d: extract_series(d, k))
    df[f"gt_{k}_series"]   = df[GT_COL].apply(lambda d: extract_series(d, k))

# Optional: extract timestamp series too
for k in KEYS:
    df[f"pred_{k}_ts_series"] = df[PRED_COL].apply(lambda d: extract_timestamp_series(d, k))
    df[f"gt_{k}_ts_series"]   = df[GT_COL].apply(lambda d: extract_timestamp_series(d, k))

# ----------------------------
# (Optional) Convenience: scalar value if exactly one step
# ----------------------------
def single_or_nan(xs):
    return xs[0] if isinstance(xs, list) and len(xs) == 1 else np.nan

for k in KEYS:
    df[f"pred_{k}_single"] = df[f"pred_{k}_series"].apply(single_or_nan)
    df[f"gt_{k}_single"]   = df[f"gt_{k}_series"].apply(single_or_nan)

# ----------------------------
# Diagnostics
# ----------------------------
print("Non-empty series counts:")
for k in KEYS:
    n_pred = (df[f"pred_{k}_series"].apply(len) > 0).sum()
    n_gt   = (df[f"gt_{k}_series"].apply(len) > 0).sum()
    print(f"  {k}: pred={n_pred} rows, gt={n_gt} rows")

print("\nPreview of extracted columns:")
cols_preview = []
for k in KEYS:
    cols_preview += [f"pred_{k}_series", f"gt_{k}_series"]
print(df[cols_preview].head(5))

# ----------------------------
# Save
# ----------------------------
df.to_csv(OUT_PATH, index=False)
print("\nSaved:", OUT_PATH)


Non-empty series counts:
  TOTALDEMAND: pred=45 rows, gt=45 rows
  RRP: pred=45 rows, gt=45 rows

Preview of extracted columns:
                             pred_TOTALDEMAND_series  \
0  [7180.0, 6920.0, 6750.0, 6640.0, 6580.0, 6610....   
1  [7340.0, 7520.0, 8170.0, 10850.0, 10320.0, 910...   
2  [6480.0, 6210.0, 6070.0, 5920.0, 5850.0, 5890....   
3  [5230.0, 6850.0, 6420.0, 5980.0, 5640.0, 5380....   
4  [5850.0, 5720.0, 6180.0, 6420.0, 5280.0, 4850....   

                               gt_TOTALDEMAND_series  \
0  [7460.3, 7439.96, 7481.72, 7254.16, 7140.31, 7...   
1  [6852.73, 6916.52, 8493.62, 10584.88, 10157.37...   
2  [7760.39, 7111.17, 6580.25, 6172.09, 6034.17, ...   
3  [6579.05, 6625.57, 6036.98, 5610.6, 5348.19, 5...   
4  [5344.69, 5372.14, 5866.64, 5030.77, 3868.89, ...   

                                     pred_RRP_series  \
0  [58.5, 54.2, 48.9, 42.3, 38.7, 36.5, 41.8, 52....   
1  [89.0, 95.0, 128.0, 300.0, 215.0, 149.0, 94.0,...   
2  [108.0, 100.0, 94.0, 86.0, 

In [2]:
print(df[cols_preview])

                              pred_TOTALDEMAND_series  \
0   [7180.0, 6920.0, 6750.0, 6640.0, 6580.0, 6610....   
1   [7340.0, 7520.0, 8170.0, 10850.0, 10320.0, 910...   
2   [6480.0, 6210.0, 6070.0, 5920.0, 5850.0, 5890....   
3   [5230.0, 6850.0, 6420.0, 5980.0, 5640.0, 5380....   
4   [5850.0, 5720.0, 6180.0, 6420.0, 5280.0, 4850....   
5   [7180.0, 6920.0, 6750.0, 6680.0, 6820.0, 7050....   
6   [7340.0, 7520.0, 8180.0, 10850.0, 10320.0, 910...   
7   [6280.0, 6140.0, 6020.0, 5890.0, 5780.0, 5720....   
8   [5240.0, 5580.0, 4850.0, 4320.0, 3980.0, 3750....   
9   [5490.0, 5220.0, 5570.0, 5570.0, 4560.0, 4500....   
10  [7180.0, 6920.0, 6750.0, 6640.0, 6580.0, 6610....   
11  [7340.0, 7520.0, 8180.0, 10850.0, 10320.0, 910...   
12  [6330.0, 6110.0, 5890.0, 5730.0, 5640.0, 5610....   
13  [5180.0, 6850.0, 6420.0, 6080.0, 5740.0, 5580....   
14  [5290.0, 5180.0, 5420.0, 5570.0, 4560.0, 4500....   
15  [7180.0, 6780.0, 6700.0, 6590.0, 6600.0, 6850....   
16  [7340.0, 7520.0, 8180.0, 10

In [3]:
import pandas as pd
import numpy as np
import ast

IN_PATH = "Gemini_multi_step.csv"
OUT_PATH = "Gemini_multi_step_extracted.csv"

PRED_COL = "predicted"
GT_COL   = "ground_truth"

KEYS = ["TOTALDEMAND", "RRP"]


def safe_to_dict(x):
    """Convert a cell to dict safely (handles dict, stringified dict, NaN)."""
    if isinstance(x, dict):
        return x
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return {}
        try:
            val = ast.literal_eval(x)
            return val if isinstance(val, dict) else {}
        except Exception:
            return {}
    return {}


def extract_series(d, key):
    """
    Extract all numeric first elements from dict[key] where dict[key] can be:
      - [value, ts]
      - [[value, ts]]
      - [[v1, ts], [v2, ts], ...]
    Returns: list[float] (possibly empty)
    """
    if not isinstance(d, dict) or key not in d:
        return []

    v = d[key]

    # Case A: [value, ts]
    if isinstance(v, (list, tuple)) and len(v) >= 1 and not isinstance(v[0], (list, tuple)):
        # v[0] is the numeric value
        try:
            return [float(v[0])]
        except Exception:
            return []

    # Case B/C: [[value, ts], [value, ts], ...]
    if isinstance(v, (list, tuple)):
        out = []
        for item in v:
            if isinstance(item, (list, tuple)) and len(item) >= 1:
                try:
                    out.append(float(item[0]))
                except Exception:
                    # skip non-numeric
                    pass
        return out

    return []


def extract_timestamp_series(d, key):
    """
    Optional: extract all timestamps (2nd element) if present.
    Returns: list[str]
    """
    if not isinstance(d, dict) or key not in d:
        return []

    v = d[key]

    # [value, ts]
    if isinstance(v, (list, tuple)) and len(v) >= 2 and not isinstance(v[0], (list, tuple)):
        return [str(v[1])]

    # [[value, ts], ...]
    if isinstance(v, (list, tuple)):
        out = []
        for item in v:
            if isinstance(item, (list, tuple)) and len(item) >= 2:
                out.append(str(item[1]))
        return out

    return []


# ----------------------------
# Load
# ----------------------------
df = pd.read_csv(IN_PATH)

# Ensure dicts
df[PRED_COL] = df[PRED_COL].apply(safe_to_dict)
df[GT_COL]   = df[GT_COL].apply(safe_to_dict)

# ----------------------------
# Extract series (values)
# ----------------------------
for k in KEYS:
    df[f"pred_{k}_series"] = df[PRED_COL].apply(lambda d: extract_series(d, k))
    df[f"gt_{k}_series"]   = df[GT_COL].apply(lambda d: extract_series(d, k))

# Optional: extract timestamp series too
for k in KEYS:
    df[f"pred_{k}_ts_series"] = df[PRED_COL].apply(lambda d: extract_timestamp_series(d, k))
    df[f"gt_{k}_ts_series"]   = df[GT_COL].apply(lambda d: extract_timestamp_series(d, k))

# ----------------------------
# (Optional) Convenience: scalar value if exactly one step
# ----------------------------
def single_or_nan(xs):
    return xs[0] if isinstance(xs, list) and len(xs) == 1 else np.nan

for k in KEYS:
    df[f"pred_{k}_single"] = df[f"pred_{k}_series"].apply(single_or_nan)
    df[f"gt_{k}_single"]   = df[f"gt_{k}_series"].apply(single_or_nan)

# ----------------------------
# Diagnostics
# ----------------------------
print("Non-empty series counts:")
for k in KEYS:
    n_pred = (df[f"pred_{k}_series"].apply(len) > 0).sum()
    n_gt   = (df[f"gt_{k}_series"].apply(len) > 0).sum()
    print(f"  {k}: pred={n_pred} rows, gt={n_gt} rows")

print("\nPreview of extracted columns:")
cols_preview = []
for k in KEYS:
    cols_preview += [f"pred_{k}_series", f"gt_{k}_series"]
print(df[cols_preview].head(5))

# ----------------------------
# Save
# ----------------------------
df.to_csv(OUT_PATH, index=False)
print("\nSaved:", OUT_PATH)


Non-empty series counts:
  TOTALDEMAND: pred=112 rows, gt=117 rows
  RRP: pred=112 rows, gt=117 rows

Preview of extracted columns:
                             pred_TOTALDEMAND_series  \
0  [9137.85, 8664.44, 8292.13, 7944.92, 7743.73, ...   
1  [6390.0, 6050.0, 5730.0, 5600.0, 5160.0, 4820....   
2  [10140.68, 8509.82, 7588.03, 8445.91, 9531.03,...   
3   [9138.0, 8664.0, 8292.0, 7945.0, 7744.0, 7532.0]   
4  [6388.715, 6051.27, 5731.685, 5597.3, 5158.83,...   

                               gt_TOTALDEMAND_series  \
0  [8716.41, 8325.79, 7859.03, 7547.44, 7261.27, ...   
1  [7226.13, 6784.28, 6100.63, 5791.05, 5398.78, ...   
2  [10933.33, 10158.23, 8945.52, 9332.74, 11310.2...   
3  [8716.41, 8325.79, 7859.03, 7547.44, 7261.27, ...   
4  [7226.13, 6784.28, 6100.63, 5791.05, 5398.78, ...   

                                     pred_RRP_series  \
0    [286.49, 175.02, 175.02, 146.83, 176.3, 144.79]   
1  [266.0, 250.0, 186.0, 243.0, 250.0, 241.0, 192...   
2  [61.08, 64.99, 77.485, 

In [4]:
import pandas as pd
import numpy as np
import ast

IN_PATH = "OpenAI_multi_step.csv"
OUT_PATH = "OpenAI_multi_step_extracted.csv"

PRED_COL = "predicted"
GT_COL   = "ground_truth"

KEYS = ["TOTALDEMAND", "RRP"]


def safe_to_dict(x):
    """Convert a cell to dict safely (handles dict, stringified dict, NaN)."""
    if isinstance(x, dict):
        return x
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return {}
        try:
            val = ast.literal_eval(x)
            return val if isinstance(val, dict) else {}
        except Exception:
            return {}
    return {}


def extract_series(d, key):
    """
    Extract all numeric first elements from dict[key] where dict[key] can be:
      - [value, ts]
      - [[value, ts]]
      - [[v1, ts], [v2, ts], ...]
    Returns: list[float] (possibly empty)
    """
    if not isinstance(d, dict) or key not in d:
        return []

    v = d[key]

    # Case A: [value, ts]
    if isinstance(v, (list, tuple)) and len(v) >= 1 and not isinstance(v[0], (list, tuple)):
        # v[0] is the numeric value
        try:
            return [float(v[0])]
        except Exception:
            return []

    # Case B/C: [[value, ts], [value, ts], ...]
    if isinstance(v, (list, tuple)):
        out = []
        for item in v:
            if isinstance(item, (list, tuple)) and len(item) >= 1:
                try:
                    out.append(float(item[0]))
                except Exception:
                    # skip non-numeric
                    pass
        return out

    return []


def extract_timestamp_series(d, key):
    """
    Optional: extract all timestamps (2nd element) if present.
    Returns: list[str]
    """
    if not isinstance(d, dict) or key not in d:
        return []

    v = d[key]

    # [value, ts]
    if isinstance(v, (list, tuple)) and len(v) >= 2 and not isinstance(v[0], (list, tuple)):
        return [str(v[1])]

    # [[value, ts], ...]
    if isinstance(v, (list, tuple)):
        out = []
        for item in v:
            if isinstance(item, (list, tuple)) and len(item) >= 2:
                out.append(str(item[1]))
        return out

    return []


# ----------------------------
# Load
# ----------------------------
df = pd.read_csv(IN_PATH)

# Ensure dicts
df[PRED_COL] = df[PRED_COL].apply(safe_to_dict)
df[GT_COL]   = df[GT_COL].apply(safe_to_dict)

# ----------------------------
# Extract series (values)
# ----------------------------
for k in KEYS:
    df[f"pred_{k}_series"] = df[PRED_COL].apply(lambda d: extract_series(d, k))
    df[f"gt_{k}_series"]   = df[GT_COL].apply(lambda d: extract_series(d, k))

# Optional: extract timestamp series too
for k in KEYS:
    df[f"pred_{k}_ts_series"] = df[PRED_COL].apply(lambda d: extract_timestamp_series(d, k))
    df[f"gt_{k}_ts_series"]   = df[GT_COL].apply(lambda d: extract_timestamp_series(d, k))

# ----------------------------
# (Optional) Convenience: scalar value if exactly one step
# ----------------------------
def single_or_nan(xs):
    return xs[0] if isinstance(xs, list) and len(xs) == 1 else np.nan

for k in KEYS:
    df[f"pred_{k}_single"] = df[f"pred_{k}_series"].apply(single_or_nan)
    df[f"gt_{k}_single"]   = df[f"gt_{k}_series"].apply(single_or_nan)

# ----------------------------
# Diagnostics
# ----------------------------
print("Non-empty series counts:")
for k in KEYS:
    n_pred = (df[f"pred_{k}_series"].apply(len) > 0).sum()
    n_gt   = (df[f"gt_{k}_series"].apply(len) > 0).sum()
    print(f"  {k}: pred={n_pred} rows, gt={n_gt} rows")

print("\nPreview of extracted columns:")
cols_preview = []
for k in KEYS:
    cols_preview += [f"pred_{k}_series", f"gt_{k}_series"]
print(df[cols_preview])

# ----------------------------
# Save
# ----------------------------
df.to_csv(OUT_PATH, index=False)
print("\nSaved:", OUT_PATH)


Non-empty series counts:
  TOTALDEMAND: pred=112 rows, gt=112 rows
  RRP: pred=105 rows, gt=112 rows

Preview of extracted columns:
                               pred_TOTALDEMAND_series  \
0             [7097.0, 7097.0, 7097.0, 7097.0, 7097.0]   
1    [4501.33, 4248.62, 4037.37, 3962.89, 3929.73, ...   
2    [8434.0, 8312.0, 8302.0, 8273.0, 8284.0, 8300....   
3             [7097.0, 7097.0, 7097.0, 7097.0, 7097.0]   
4    [5404.13, 5346.6, 5240.84, 5239.06, 5226.1, 52...   
..                                                 ...   
107  [7258.0, 7260.0, 7260.0, 7260.0, 7260.0, 7260....   
108  [7097.0, 7076.0, 6908.0, 6671.0, 6509.0, 6383....   
109  [6056.0, 6056.0, 6056.0, 6056.0, 6056.0, 6056....   
110  [4501.33, 4248.62, 4037.37, 3962.89, 3929.73, ...   
111  [5633.0, 5614.0, 5585.0, 5560.0, 5540.0, 5520....   

                                 gt_TOTALDEMAND_series  \
0    [8716.41, 8325.79, 7859.03, 7547.44, 7261.27, ...   
1    [7226.13, 6784.28, 6100.63, 5791.05, 5398.78, ... 

In [5]:
import pandas as pd
import numpy as np
import ast

IN_PATH = "DeepSeek_multi_step.csv"
OUT_PATH = "DeepSeek_multi_step_extracted.csv"

PRED_COL = "predicted"
GT_COL   = "ground_truth"

KEYS = ["TOTALDEMAND", "RRP"]


def safe_to_dict(x):
    """Convert a cell to dict safely (handles dict, stringified dict, NaN)."""
    if isinstance(x, dict):
        return x
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return {}
        try:
            val = ast.literal_eval(x)
            return val if isinstance(val, dict) else {}
        except Exception:
            return {}
    return {}


def extract_series(d, key):
    """
    Extract all numeric first elements from dict[key] where dict[key] can be:
      - [value, ts]
      - [[value, ts]]
      - [[v1, ts], [v2, ts], ...]
    Returns: list[float] (possibly empty)
    """
    if not isinstance(d, dict) or key not in d:
        return []

    v = d[key]

    # Case A: [value, ts]
    if isinstance(v, (list, tuple)) and len(v) >= 1 and not isinstance(v[0], (list, tuple)):
        # v[0] is the numeric value
        try:
            return [float(v[0])]
        except Exception:
            return []

    # Case B/C: [[value, ts], [value, ts], ...]
    if isinstance(v, (list, tuple)):
        out = []
        for item in v:
            if isinstance(item, (list, tuple)) and len(item) >= 1:
                try:
                    out.append(float(item[0]))
                except Exception:
                    # skip non-numeric
                    pass
        return out

    return []


def extract_timestamp_series(d, key):
    """
    Optional: extract all timestamps (2nd element) if present.
    Returns: list[str]
    """
    if not isinstance(d, dict) or key not in d:
        return []

    v = d[key]

    # [value, ts]
    if isinstance(v, (list, tuple)) and len(v) >= 2 and not isinstance(v[0], (list, tuple)):
        return [str(v[1])]

    # [[value, ts], ...]
    if isinstance(v, (list, tuple)):
        out = []
        for item in v:
            if isinstance(item, (list, tuple)) and len(item) >= 2:
                out.append(str(item[1]))
        return out

    return []


# ----------------------------
# Load
# ----------------------------
df = pd.read_csv(IN_PATH)

# Ensure dicts
df[PRED_COL] = df[PRED_COL].apply(safe_to_dict)
df[GT_COL]   = df[GT_COL].apply(safe_to_dict)

# ----------------------------
# Extract series (values)
# ----------------------------
for k in KEYS:
    df[f"pred_{k}_series"] = df[PRED_COL].apply(lambda d: extract_series(d, k))
    df[f"gt_{k}_series"]   = df[GT_COL].apply(lambda d: extract_series(d, k))

# Optional: extract timestamp series too
for k in KEYS:
    df[f"pred_{k}_ts_series"] = df[PRED_COL].apply(lambda d: extract_timestamp_series(d, k))
    df[f"gt_{k}_ts_series"]   = df[GT_COL].apply(lambda d: extract_timestamp_series(d, k))

# ----------------------------
# (Optional) Convenience: scalar value if exactly one step
# ----------------------------
def single_or_nan(xs):
    return xs[0] if isinstance(xs, list) and len(xs) == 1 else np.nan

for k in KEYS:
    df[f"pred_{k}_single"] = df[f"pred_{k}_series"].apply(single_or_nan)
    df[f"gt_{k}_single"]   = df[f"gt_{k}_series"].apply(single_or_nan)

# ----------------------------
# Diagnostics
# ----------------------------
print("Non-empty series counts:")
for k in KEYS:
    n_pred = (df[f"pred_{k}_series"].apply(len) > 0).sum()
    n_gt   = (df[f"gt_{k}_series"].apply(len) > 0).sum()
    print(f"  {k}: pred={n_pred} rows, gt={n_gt} rows")

print("\nPreview of extracted columns:")
cols_preview = []
for k in KEYS:
    cols_preview += [f"pred_{k}_series", f"gt_{k}_series"]
print(df[cols_preview].head(5))

# ----------------------------
# Save
# ----------------------------
df.to_csv(OUT_PATH, index=False)
print("\nSaved:", OUT_PATH)


Non-empty series counts:
  TOTALDEMAND: pred=117 rows, gt=117 rows
  RRP: pred=117 rows, gt=117 rows

Preview of extracted columns:
                             pred_TOTALDEMAND_series  \
0   [8660.0, 8290.0, 7940.0, 7740.0, 7530.0, 7280.0]   
1  [6390.0, 6050.0, 5730.0, 5600.0, 5160.0, 4820....   
2  [9490.0, 8510.0, 7590.0, 7780.0, 9530.0, 11300...   
3   [8660.0, 8290.0, 7940.0, 7740.0, 7530.0, 7280.0]   
4  [6310.0, 6050.0, 5730.0, 5600.0, 5220.0, 4600....   

                               gt_TOTALDEMAND_series  \
0  [8716.41, 8325.79, 7859.03, 7547.44, 7261.27, ...   
1  [7226.13, 6784.28, 6100.63, 5791.05, 5398.78, ...   
2  [10933.33, 10158.23, 8945.52, 9332.74, 11310.2...   
3  [8716.41, 8325.79, 7859.03, 7547.44, 7261.27, ...   
4  [7226.13, 6784.28, 6100.63, 5791.05, 5398.78, ...   

                                     pred_RRP_series  \
0         [176.0, 146.0, 118.0, 116.0, 145.0, 140.0]   
1  [266.0, 250.0, 186.0, 243.0, 241.0, 192.0, 184...   
2  [91.0, 65.0, 77.0, 52.0